# PAXRAW Data Processing Pipeline

Fast processing of NHANES accelerometer data using numba-optimized algorithms.

**Inputs:** Raw PAXRAW data (parquet, xpt, or zip)

**Outputs:**
- `paxraw_{c/d}.parquet` - Raw data in parquet format
- `paxraw_{c/d}_met.parquet` - With MET values
- `paxraw_{c/d}_met_worn.parquet` - With worn indicator
- `paxraw_{c/d}_met_worn_bouts.parquet` - With bout classifications
- `paxraw_{c/d}_met_worn_bouts_reliable.parquet` - Filtered to reliable data
- Various summary files (people, people_days, etc.)

## Imports

In [ ]:
import subprocess

import numpy as np
import pandas as pd

from utils import (
    CHAR_LOOKUP,
    flatten_columns,
    get_datadir,
    get_processed_datadir,
    worn_indicator_numba,
    bout_classifier,
)

## Parameters

In [ ]:
year: str = "2003-2004"

In [ ]:
datadir = get_datadir(year)
processeddir = get_processed_datadir(year)
print(f"Raw data: {datadir}")
print(f"Processed data: {processeddir}")

## Load Raw Data

In [ ]:
parquetfile = datadir / f"paxraw_{CHAR_LOOKUP[year].lower()}.parquet"
xptfile = datadir / f"paxraw_{CHAR_LOOKUP[year].lower()}.xpt"
zipfilename = f"PAXRAW_{CHAR_LOOKUP[year].upper()}.zip"
zipfile = datadir / zipfilename

if parquetfile.exists():
    print(f"Loading parquet: {parquetfile}")
    paxraw = pd.read_parquet(parquetfile)
elif xptfile.exists():
    print(f"Loading xpt: {xptfile}")
    paxraw = pd.read_sas(xptfile)
else:
    print("No parquet or xpt file, downloading zip...")
    print("This is a really big file, I'd recommend downloading it using a broswer and unzipping it")
    if not zipfile.exists():
        subprocess.run([
            "wget", "-O", zipfile,
            f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year.split('-')[0]}/DataFiles/{zipfilename}",
            "--no-use-server-timestamps",
        ])
    print("Extracting...")
    subprocess.run(["unzip", "-o", zipfile, "-d", datadir])
    paxraw = pd.read_sas(xptfile)

print(f"Shape: {paxraw.shape}")

## Fix Datatypes and save parquet

In [ ]:
if not parquetfile.exists():  # only need to fix them if we loaded the SAS version
    for col in paxraw.columns:
        try:
            paxraw.loc[:, col] = paxraw.loc[:, col].astype(int)
        except pd.errors.IntCastingNaNError:
            paxraw.loc[:, col] = paxraw.loc[:, col].replace(np.nan, 0).astype(int)
    # Save raw parquet if we loaded from xpt
    paxraw.to_parquet(parquetfile)
    print(f"Saved: {parquetfile}")

### Don't add a datetime column, takes too long

```
paxraw["datetime"] = paxraw.progress_apply(
    lambda x: datetime.datetime(2006, 1, 1) + datetime.timedelta(
        days=int(x.PAXDAY - 1),
        hours=int(x.PAXHOUR),
        minutes=int(x.PAXMINUT)
    ),
    axis=1,
)
```

## Define intensity level cuts and METs

Cuts defined in literature and in [common software](https://github.com/vandomed/nhanesaccel/blob/7ebd7a0cd6e2f169e6f81a66c8c99b1746eacb51/R/process_nhanes.R#L267)

In [ ]:
int_cuts = [100, 760, 2020, 5999]
# add end ranges for interpolation
int_cuts_endranges = [paxraw.PAXINTEN.min()] + int_cuts + [paxraw.PAXINTEN.max() + 1]
# MET values corresponding to each cut point
METs = [1, 1, 2, 3.5, 6, 10]
labels = ["Sedentary", "Low", "Light", "Moderate", "Vigorous"]

In [ ]:
# linearly interpolate MET values
METs_full = np.interp(
    np.arange(int_cuts_endranges[0], int_cuts_endranges[-1]), int_cuts_endranges, METs
)
METs_lookup = pd.DataFrame({
    "MET": METs_full,
    "PAXINTEN": np.arange(int_cuts_endranges[0], int_cuts_endranges[-1]),
})

### Join METs

In [ ]:
paxraw = paxraw.merge(METs_lookup, how="left", on="PAXINTEN")

In [ ]:
parquetfile = processeddir / f"paxraw_{CHAR_LOOKUP[year].lower()}_met.parquet"
paxraw.to_parquet(parquetfile)
print(f"Saved: {parquetfile}")

## Worn Classification

Using numba-optimized algorithm. See `02_algorithm_development.ipynb` for comparison of approaches.

In [ ]:
paxraw["worn"] = worn_indicator_numba(paxraw.PAXINTEN.values, paxraw.SEQN.values)

In [ ]:
parquetfile = processeddir / f"paxraw_{CHAR_LOOKUP[year].lower()}_met_worn.parquet"
paxraw.to_parquet(parquetfile)
print(f"Saved: {parquetfile}")

## Bout Classification

Classify continuous bouts of activity at each intensity level.

In [ ]:
paxraw["vigorous_bout"] = bout_classifier(
    paxraw.PAXINTEN.values,
    paxraw.SEQN.values,
    paxraw.worn.values,
    np.zeros(paxraw.PAXINTEN.values.shape[0]),
    upper=int_cuts_endranges[5],
    lower=int_cuts_endranges[4],
    tol_upper_soft=0,
    tol_lower_soft=0,
    m=10,
    lower_soft=(int_cuts_endranges[3] + int_cuts_endranges[4]) / 2,
    check_already_classified=False,
)

In [ ]:
paxraw["moderate_bout"] = bout_classifier(
    paxraw.PAXINTEN.values,
    paxraw.SEQN.values,
    paxraw.worn.values,
    paxraw.vigorous_bout.values,
    upper=int_cuts_endranges[4],
    lower=int_cuts_endranges[3],
    tol_upper_soft=10,
    tol_lower_soft=0,
    m=10,
    lower_soft=(int_cuts_endranges[2] + int_cuts_endranges[3]) / 2,
    upper_soft=int_cuts_endranges[5],
    check_already_classified=True,
)

In [ ]:
paxraw["light_bout"] = bout_classifier(
    paxraw.PAXINTEN.values,
    paxraw.SEQN.values,
    paxraw.worn.values,
    np.maximum(paxraw.moderate_bout.values, paxraw.vigorous_bout.values),
    upper=int_cuts_endranges[3],
    lower=int_cuts_endranges[2],
    tol_upper_soft=10,
    tol_lower_soft=0,
    m=10,
    lower_soft=(int_cuts_endranges[1] + int_cuts_endranges[2]) / 2,
    upper_soft=int_cuts_endranges[5],
    check_already_classified=True,
)

In [ ]:
paxraw["low_bout"] = bout_classifier(
    paxraw.PAXINTEN.values,
    paxraw.SEQN.values,
    paxraw.worn.values,
    np.maximum(paxraw.light_bout.values, paxraw.moderate_bout.values, paxraw.vigorous_bout.values),
    upper=int_cuts_endranges[2],
    lower=int_cuts_endranges[1],
    tol_upper_soft=10,
    tol_lower_soft=0,
    m=10,
    lower_soft=(int_cuts_endranges[0] + int_cuts_endranges[1]) / 2,
    upper_soft=int_cuts_endranges[5],
    check_already_classified=True,
)

In [ ]:
paxraw["sed_bout"] = bout_classifier(
    paxraw.PAXINTEN.values,
    paxraw.SEQN.values,
    paxraw.worn.values,
    paxraw.low_bout.values,
    upper=int_cuts_endranges[1],
    lower=int_cuts_endranges[0],
    tol_upper_soft=0,
    tol_lower_soft=0,
    m=10,
    check_already_classified=False,
)

In [ ]:
paxraw["no_bout"] = (
    (paxraw["worn"] == 1)
    & (paxraw["sed_bout"] == 0)
    & (paxraw["low_bout"] == 0)
    & (paxraw["light_bout"] == 0)
    & (paxraw["moderate_bout"] == 0)
    & (paxraw["vigorous_bout"] == 0)
) * 1

In [ ]:
# A single column to label minute-by-minute intensity
paxraw["intensity"] = pd.cut(
    paxraw.PAXINTEN.values, int_cuts_endranges, right=False, labels=range(len(labels))
)
# don't include the labels for size:
# labels=labels
paxraw["METh"] = paxraw.MET / 60
paxraw["activeMETh"] = (paxraw.MET - 1) / 60

In [ ]:
# save parquet
parquetfile = processeddir / f"paxraw_{CHAR_LOOKUP[year].lower()}_met_worn_bouts.parquet"
paxraw.to_parquet(parquetfile)
print(f"Saved: {parquetfile}")

## Valid Days & Filtering

In [ ]:
min_worn_hours_threshold = 10
MINUTES_PER_HOUR = 60

In [ ]:
# sum minutes of wear and activity counts per day
worn_minutes = paxraw.groupby(["SEQN", "PAXDAY"]).agg({"worn": ["sum"]})
# compute valid days
worn_minutes["valid_day"] = (
    worn_minutes["worn"]["sum"] > (min_worn_hours_threshold * MINUTES_PER_HOUR)
) * 1
worn_minutes.columns = flatten_columns(worn_minutes.columns.values)

In [ ]:
paxraw["max_intensity"] = (paxraw.PAXINTEN == 32767) * 1
paxraw["out_of_calibration"] = (paxraw.PAXCAL == 2) * 1
paxraw["unreliable"] = (paxraw.PAXSTAT == 2) * 1

if "PAXSTEP" not in paxraw.columns:
    paxraw["PAXSTEP"] = 0
    paxraw["zero_steps_with_intensity"] = 0
else:
    paxraw["zero_steps_with_intensity"] = ((paxraw.PAXINTEN > 250) & (paxraw.PAXSTEP == 0)) * 1

paxraw["too_many_steps"] = (paxraw.PAXSTEP > 200) * 1
paxraw["steps_filtered_500"] = 0
paxraw.loc[paxraw.PAXINTEN >= 500, "steps_filtered_500"] = paxraw.PAXSTEP
paxraw["steps_filtered_300"] = 0
paxraw.loc[paxraw.PAXINTEN >= 300, "steps_filtered_300"] = paxraw.PAXSTEP

In [ ]:
agg_columns = [
    "PAXINTEN", "max_intensity", "out_of_calibration", "unreliable",
    "zero_steps_with_intensity", "too_many_steps", "steps_filtered_500", "steps_filtered_300"
]
tudor2009_filters = (
    paxraw.groupby(["SEQN", "PAXDAY"])
    .agg({col: ["sum", "last"] for col in agg_columns})
    .reset_index()
)
tudor2009_filters.columns = flatten_columns(tudor2009_filters.columns.values)

### Join all person-day level indicators

In [ ]:
d_people_days = tudor2009_filters.merge(worn_minutes, how="inner", on=["SEQN", "PAXDAY"])

In [ ]:
parquetfile = processeddir / f"paxraw_{CHAR_LOOKUP[year].lower()}_people_days.parquet"
d_people_days.to_parquet(parquetfile)
print(f"Saved: {parquetfile}")

### Sum up to person level

In [ ]:
d_people = d_people_days.groupby("SEQN").agg({
    "zero_steps_with_intensity_sum": "sum",
    "too_many_steps_sum": "sum",
    "max_intensity_sum": "sum",
    "out_of_calibration_sum": "sum",
    "out_of_calibration_last": "last",
    "unreliable_sum": "sum",
    "unreliable_last": "last",
    "steps_filtered_500_sum": "mean",
    "steps_filtered_300_sum": "mean",
    "valid_day": "sum",
    "PAXINTEN_sum": "mean",
})

In [ ]:
parquetfile = processeddir / f"paxraw_{CHAR_LOOKUP[year].lower()}_people.parquet"
d_people.to_parquet(parquetfile)
print(f"Saved: {parquetfile}")

In [ ]:
d_reliable = d_people.loc[
    (d_people.zero_steps_with_intensity_sum <= 10)
    & (d_people.too_many_steps_sum <= 10)
    & (d_people.max_intensity_sum <= 10)
    & (~d_people.out_of_calibration_last)
    & (d_people.unreliable_sum <= 10)
    & (d_people.steps_filtered_500_sum <= 200000),
    :,
]
print(f"Reliable people: {d_reliable.shape[0]} / {d_people.shape[0]}")

In [ ]:
parquetfile = processeddir / f"paxraw_{CHAR_LOOKUP[year].lower()}_reliable_people.parquet"
d_reliable.to_parquet(parquetfile)
print(f"Saved: {parquetfile}")

In [ ]:
paxraw_reliable = paxraw.merge(
    worn_minutes.loc[worn_minutes.valid_day == 1, :], on=["SEQN", "PAXDAY"]
).merge(d_reliable.loc[:, []], how="inner", on="SEQN")
print(f"Reliable data: {paxraw_reliable.shape[0]:,} rows")

In [ ]:
parquetfile = processeddir / f"paxraw_{CHAR_LOOKUP[year].lower()}_met_worn_bouts_reliable.parquet"
paxraw_reliable.to_parquet(parquetfile)
print(f"Saved: {parquetfile}")

## Fishman Summary

Per-person summary of activity counts following Fishman (2016) methodology.

In [ ]:
person_active_counts = d_people_days.loc[
    :, ["SEQN", "worn_sum", "PAXINTEN_sum", "valid_day"]
].copy()
person_active_counts.columns = ["SEQN", "worn_minutes", "activity_counts", "valid_day"]

person_active_counts_summary = (
    person_active_counts.loc[person_active_counts.valid_day == 1, :]
    .groupby(["SEQN"])
    .agg({"worn_minutes": "mean", "activity_counts": "mean", "valid_day": "count"})
)
person_active_counts_summary.columns = [
    "daily_worn_minutes_mean",
    "daily_activity_count_sum_mean",
    "n_valid_days",
]

In [ ]:
parquetfile = processeddir / f"paxraw_{CHAR_LOOKUP[year].lower()}_fishman_summary.parquet"
person_active_counts_summary.to_parquet(parquetfile)
print(f"Saved: {parquetfile}")

## Done

All processed files saved to the processed directory.

In [ ]:
print("Output files:")
for f in sorted(processeddir.glob("paxraw_*.parquet")):
    print(f"  {f.name}: {f.stat().st_size / 1024 / 1024:.1f} MB")